# Study 946 — Distribution is not Return 💸

**A high-payout ETF advertises a big distribution rate. Does the big number predict a big
return?**

Fifteen listed income funds — QYLD, XYLD, RYLD, JEPI, JEPQ, SPYI, DIVO, NUSI, PBP, PFF and
five dividend-equity funds — are ranked every month on their **trailing-12-month
distribution rate**, reconstructed from the gap between the total-return tape and the
price-only tape. The top third is bought, the bottom third sold, and the pair is held for
the following month (**one execution lag**). We then ask what that rank predicted: the next
payout, the next *price* move, or the next *total* return.

Held months **2013-11-30 → 2026-06-30** (152 months, cross-section
6–15 funds). Every real number below is frozen from
`docs/results.md` (return fingerprints `ac334d204e26` / `b94d79082051`); the only live cells
run the **offline synthetic** control and are labelled as such. As-of 2026-06-30.

One thing to keep in view from the first line: the payout is *measured* as the gap between
the two tapes, so "price return" is by definition "total return minus payout". The two
independent facts on this page are how forecastable the **payout** is and how unforecastable
the **total return** is; the price erosion everyone quotes is what those two imply.


## 1. What a distribution actually is

When a fund pays you $1, the fund is worth $1 less. The cash does not appear from nowhere — it comes out of the pot you own a share of. So a "12% distribution rate" is a statement about *how the money is packaged*, not about how much money there is.

That gives us a clean test. Every fund has two price series: the **total-return** one (which pretends you reinvested every payment) and the **price-only** one (the quote you actually see). The gap between them is exactly what was paid out. So we can reconstruct each fund's payout rate — and then check what a big one predicts.

> 🔬 *For the quants:* the reconstruction is `(1+r_total)/(1+r_price) − 1` per month, compounded over twelve. It lands within a few tenths of a point of every fund's published sticker, which is the check that the gap really is the distribution.

## 2. Look at the two columns side by side

Before any statistics, just read the tape. The funds with the fattest payouts have **negative price CAGRs** — their quoted share price has been sinking for a decade. The funds with the smallest payouts have the fastest-rising prices.

In [1]:
rows = [
    ('QYLD', 11.2, -2.58, 8.41),
    ('RYLD', 12.5, -6.16, 5.47),
    ('PFF',   6.5, -2.55, 3.69),
    ('SCHD',  3.2, 9.39, 12.93),
    ('NOBL',  2.1, 7.94, 10.21),
]
print(f"{'fund':6s}{'payout':>9s}{'price CAGR':>13s}{'total CAGR':>13s}")
for name, pay, px, tot in rows:
    print(f'{name:6s}{pay:8.1f}%{px:12.2f}%{tot:12.2f}%')
print()
print('The price column sorts itself by payout. The total column does not.')

fund     payout   price CAGR   total CAGR
QYLD      11.2%       -2.58%        8.41%
RYLD      12.5%       -6.16%        5.47%
PFF        6.5%       -2.55%        3.69%
SCHD       3.2%        9.39%       12.93%
NOBL       2.1%        7.94%       10.21%

The price column sorts itself by payout. The total column does not.


## 3. The three questions, answered

Rank the whole universe on payout each month and see what the rank forecasts about the month that follows. Three different things to forecast, three very different answers.

In [2]:
R = dict(fm_dist=24.6, fm_dist_t=11.27, fm_price=-28.1, fm_price_t=-3.84,
         fm_total=-3.5, fm_total_t=-0.48)
print('what the payout rank predicts about NEXT MONTH (bps per 1sd of payout):')
print('  the next payout      : %+6.1f bps   t = %+6.2f   <- almost perfectly forecastable'
      % (R['fm_dist'], R['fm_dist_t']))
print('  the PRICE move       : %+6.1f bps   t = %+6.2f   <- reliably downward'
      % (R['fm_price'], R['fm_price_t']))
print('  the TOTAL return     : %+6.1f bps   t = %+6.2f   <- nothing at all'
      % (R['fm_total'], R['fm_total_t']))

what the payout rank predicts about NEXT MONTH (bps per 1sd of payout):
  the next payout      :  +24.6 bps   t = +11.27   <- almost perfectly forecastable
  the PRICE move       :  -28.1 bps   t =  -3.84   <- reliably downward
  the TOTAL return     :   -3.5 bps   t =  -0.48   <- nothing at all


## 4. The give-back ratio

Buy the top third by payout and sell the bottom third. The pair collects an extra **51.6 bps a month** of distributions — a **7.05 percentage-point** payout spread (9.92% versus 2.87%). And it gives back **69.9 bps a month** in price.

That is a **give-back ratio of 1.36**. Certainly not zero — the free-money reading is dead. Whether it is genuinely *above* one is a different question, and the honest answer is that we cannot tell: what is left over — the total return — is **-18.3 bps a month with a *t* of -1.24**, statistically indistinguishable from nothing, and the gap between 1.36 and a clean 1.00 *is* that leftover. Read it as "one, and no evidence of better".

> 🔬 *For the quants:* the payout is defined as the total/price gap, so the three legs obey `price = total − payout` exactly (correlation 0.99995). The erosion leg's bootstrap CI is [-100.5, -40.4] bps and the total leg's is [-47.2, +10.4] — i.e. **[-5.67%, +1.25%] a year**, which rules out the sales pitch and rules out nothing on the downside.

## 5. So is the fat-payout basket a bad buy?

Not catastrophically — it is just *unremarkable*. Over these 152 months the high-payout third compounded at **9.06%/yr**, the low-payout third at **11.07%**, and plain SPY at **14.02%**. Adjusted for how much market risk each one carries, the high-payout basket is a **0.65-beta equity position with no alpha** (-5.3 bps/month, *t* = -0.59).

You are not being robbed. You are being sold a number that does not mean what the fact sheet implies it means.

## 6. Live check — the machinery is honest (offline synthetic)

**This cell runs a simulation, not the real tape.** We build a fake fund universe twice. In the first, the payout is pure return of capital: the estimator must find the erosion and find *nothing* in total return. In the second we secretly plant a real payout-to-return link: the estimator must find it. If it passes both, the flat real-tape answer is a fact about income ETFs, not a broken harness.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from dist_illusion import data, strategy as st
null,  _     = data.synthetic_panel(signal_strength=0.0, seed=946)
plant, truth = data.synthetic_panel(signal_strength=1.0, seed=946)
dn, dp = st.synthetic_detect(null), st.synthetic_detect(plant)
print('SYNTHETIC (not the real tape)')
print('  pure return-of-capital world: total-return slope %+.1f bps (t %+.2f)  '
      'price slope %+.1f (t %+.2f)  give-back %.2f'
      % (dn['fm_total_bps'], dn['t_total'], dn['fm_price_bps'], dn['t_price'], dn['giveback']))
print('  planted-link world         : total-return slope %+.1f bps (t %+.2f)  '
      '(we planted %+.1f)'
      % (dp['fm_total_bps'], dp['t_total'], truth['planted_slope_per_sd']*1e4))

SYNTHETIC (not the real tape)
  pure return-of-capital world: total-return slope +0.3 bps (t +0.10)  price slope -28.8 (t -8.80)  give-back 0.99
  planted-link world         : total-return slope +30.2 bps (t +9.20)  (we planted +30.0)


## Verdict

- **Signal — Real.** "Distribution is not return" is not a slogan here, it is the tape's arithmetic — and the arithmetic word is doing real work. Two facts are measured: the payout rank forecasts the **next payout** (*t* = +11.27) and forecasts **total return not at all** (*t* = -1.24, CI [-5.67%, +1.25%] a year). The **price fall** (*t* = -4.53) then follows by subtraction — it is the same evidence in another column, not a third confirmation. Caveats: the erosion softens inside the buy-write cohort alone (*t* = -1.93 over 74 months), the universe only contains funds that survived, and the null bounds the sales pitch without ruling out a downside penalty.
- **Tradability — Mirage.** There is nothing to trade. Long low-payout / short high-payout earns +18.3 bps a month **gross** with *t* = 1.24 and a CI straddling zero, and the short leg pays borrow. The long-only version trails SPY on return *and* on Sharpe. Use the distribution rate as a **transparency tool** — it tells you where the cash is coming from — and never as a **signal**.